In [1]:
import os
os.chdir("C:/Users/Anwender/Downloads/Master/data_ui")


In [2]:
import pandas as pd

# PZ Pfad anpassen!!!


df = pd.read_csv("archive_data2/originalData.csv", dtype_backend ='numpy_nullable'  )
df = df.drop(columns = ["Unnamed: 0"], axis = 1)



In [3]:
df

,Year,Make,Model,Kilometres,Body Type,Engine,Transmission,Drivetrain,Exterior Colour,Interior Colour,Passengers,Doors,Fuel Type,City,Highway,Price
0,2019,Acura,MDX,53052 km,SUV,V6 Cylinder Engine,9 Speed Automatic,AWD,Majestic Black Pearl,Red,<NA>,<NA>,Gas,12.2L/100km,9.0L - 9.5L/100km,43880
1,2018,Acura,MDX,77127 km,SUV,V6 Cylinder Engine,9 Speed Automatic,AWD,Modern Steel Metallic,Black,<NA>,<NA>,Gas,12.6L/100km,9.0L/100km,36486
2,2019,Acura,RDX,33032 km,SUV,2.0L 4cyl,10 Speed Automatic,AWD,White Diamond Pearl,Black,5.0,4,Premium Unleaded,11.0L/100km,8.6L/100km,40888
3,2020,Acura,RDX,50702 km,SUV,4 Cylinder Engine,<NA>,AWD,Platinum White Pearl,Black,<NA>,<NA>,Gas,11.0L/100km,8.6L/100km,44599
4,2021,Acura,RDX,67950 km,SUV,4 Cylinder Engine,<NA>,AWD,Apex Blue Pearl,Red,<NA>,<NA>,Gas,11.3L/100km,9.1L/100km,46989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24193,2017,Volvo,S90,81000 km,Sedan,<NA>,Automatic,AWD,White,<NA>,<NA>,5 doors,Gasoline,<NA>,<NA>,34680
24194,2020,Volvo,XC40,92450 km,SUV,2.0,Automatic,AWD,Black,Black,5.0,5,Gas,<NA>,<NA>,35898
24195,2017,Volvo,XC90,92000 km,Hatchback,<NA>,Automatic,AWD,Grey,<NA>,<NA>,4 doors,Gasoline,<NA>,<NA>,38000
24196,2018,Volvo,XC90,67000 km,<NA>,<NA>,Automatic,AWD,Black,<NA>,<NA>,4 doors,Gasoline,<NA>,<NA>,45000


In [4]:
df = df.set_axis([
    'year_of_manufacture', 'manufacturer', 'model', 'mileage', 'body_type',
    'engine_size', 'transmission', 'drivetrain', 'exterior_colour', 'interior_colour',
    'passengers', 'doors', 'fuel_type', 'city', 'highway', 'price'
], axis=1)


In [5]:
df

,year_of_manufacture,manufacturer,model,mileage,body_type,engine_size,transmission,drivetrain,exterior_colour,interior_colour,passengers,doors,fuel_type,city,highway,price
0,2019,Acura,MDX,53052 km,SUV,V6 Cylinder Engine,9 Speed Automatic,AWD,Majestic Black Pearl,Red,<NA>,<NA>,Gas,12.2L/100km,9.0L - 9.5L/100km,43880
1,2018,Acura,MDX,77127 km,SUV,V6 Cylinder Engine,9 Speed Automatic,AWD,Modern Steel Metallic,Black,<NA>,<NA>,Gas,12.6L/100km,9.0L/100km,36486
2,2019,Acura,RDX,33032 km,SUV,2.0L 4cyl,10 Speed Automatic,AWD,White Diamond Pearl,Black,5.0,4,Premium Unleaded,11.0L/100km,8.6L/100km,40888
3,2020,Acura,RDX,50702 km,SUV,4 Cylinder Engine,<NA>,AWD,Platinum White Pearl,Black,<NA>,<NA>,Gas,11.0L/100km,8.6L/100km,44599
4,2021,Acura,RDX,67950 km,SUV,4 Cylinder Engine,<NA>,AWD,Apex Blue Pearl,Red,<NA>,<NA>,Gas,11.3L/100km,9.1L/100km,46989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24193,2017,Volvo,S90,81000 km,Sedan,<NA>,Automatic,AWD,White,<NA>,<NA>,5 doors,Gasoline,<NA>,<NA>,34680
24194,2020,Volvo,XC40,92450 km,SUV,2.0,Automatic,AWD,Black,Black,5.0,5,Gas,<NA>,<NA>,35898
24195,2017,Volvo,XC90,92000 km,Hatchback,<NA>,Automatic,AWD,Grey,<NA>,<NA>,4 doors,Gasoline,<NA>,<NA>,38000
24196,2018,Volvo,XC90,67000 km,<NA>,<NA>,Automatic,AWD,Black,<NA>,<NA>,4 doors,Gasoline,<NA>,<NA>,45000


In [6]:

df["exterior_colour"].unique()

<StringArray>
[         'Majestic Black Pearl',         'Modern Steel Metallic',
           'White Diamond Pearl',          'Platinum White Pearl',
               'Apex Blue Pearl',           'Crystal Black Pearl',
                        'Silver',         'Bellanova White Pearl',
                     'Dark Blue',         'Lunar Silver Metallic',
 ...
          'Pepper Gray Metallic',          'Rising Blue Metallic',
          'Terra Brown Metallic',         'Tempest Blue Metallic',
           'Deep Black Metallic', 'Deep Black Pearl w/Brown Roof',
     'Dark Mauve Pearl Metallic',             'Black Magic Pearl',
                 'Saturn Yellow',         'Terra Bronze Metallic']
Length: 1458, dtype: string

In [9]:
import numpy as np
print("--- 1. Replacing '<NA>' strings with np.nan ---")
df.replace('<NA>', np.nan, inplace=True)


# --- Step 2: Clean and Convert "Almost-Numeric" Columns ---
print("\n--- 2. Cleaning and converting data types ---")

# Clean 'mileage'
if df['mileage'].dtype == 'object':
    df['mileage'] = df['mileage'].str.replace(' km', '', regex=False).astype(float)
    print("Cleaned 'mileage' column.")

# Clean 'doors'
if df['doors'].dtype == 'object':
    df['doors'] = df['doors'].str.replace(' doors', '', regex=False).astype(float)
    print("Cleaned 'doors' column.")

# Define a function to clean fuel economy columns (city/highway)
def clean_fuel_economy(value):
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).lower().replace('l/100km', '').strip()
    
    if '-' in value_str:
        # Handle ranges like '9.0 - 9.5' by taking the average
        try:
            low, high = map(float, value_str.split('-'))
            return (low + high) / 2
        except:
            return np.nan # Return NaN if parsing fails
    else:
        # Handle single values
        try:
            return float(value_str)
        except:
            return np.nan

# Apply the function to 'city' and 'highway' columns
if df['city'].dtype == 'object':
    df['city'] = df['city'].apply(clean_fuel_economy)
    print("Cleaned 'city' column.")

if df['highway'].dtype == 'object':
    df['highway'] = df['highway'].apply(clean_fuel_economy)
    print("Cleaned 'highway' column.")


# --- Step 3: Impute Missing Values ---
print("\n--- 3. Filling missing (NaN) values ---")

# Identify numeric and categorical columns AFTER cleaning
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

# Fill numeric columns with the median
print("Filling numeric columns with median...")
for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  - Filled '{col}' with median value: {median_val}")

# Fill categorical columns with the most frequent value (mode)
print("\nFilling categorical columns with mode...")
for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0] # .mode() returns a Series, take the first item
        df[col].fillna(mode_val, inplace=True)
        print(f"  - Filled '{col}' with mode value: '{mode_val}'")


# --- Verification ---
print("\n--- 4. Verification ---")
missing_values = df.isnull().sum()
print("Missing values count per column after processing:")
print(missing_values[missing_values > 0]) # Should be empty

if missing_values.sum() == 0:
    print("\n✅ Success! The DataFrame is now complete with no missing values.")
else:
    print("\n⚠️ Warning! Some missing values still exist.")

print("\nFirst 5 rows of the cleaned DataFrame:")
print(df.head())






--- 1. Replacing '<NA>' strings with np.nan ---

--- 2. Cleaning and converting data types ---

--- 3. Filling missing (NaN) values ---
Filling numeric columns with median...

Filling categorical columns with mode...

--- 4. Verification ---
Missing values count per column after processing:
Series([], dtype: int64)

✅ Success! The DataFrame is now complete with no missing values.

First 5 rows of the cleaned DataFrame:
   year_of_manufacture manufacturer model   mileage body_type  \
0                 2019        Acura   MDX  53052 km       SUV   
1                 2018        Acura   MDX  77127 km       SUV   
2                 2019        Acura   RDX  33032 km       SUV   
3                 2020        Acura   RDX  50702 km       SUV   
4                 2021        Acura   RDX  67950 km       SUV   

          engine_size        transmission drivetrain        exterior_colour  \
0  V6 Cylinder Engine   9 Speed Automatic        AWD   Majestic Black Pearl   
1  V6 Cylinder Engine   9 Sp

In [10]:
df

,year_of_manufacture,manufacturer,model,mileage,body_type,engine_size,transmission,drivetrain,exterior_colour,interior_colour,passengers,doors,fuel_type,city,highway,price
0,2019,Acura,MDX,53052 km,SUV,V6 Cylinder Engine,9 Speed Automatic,AWD,Majestic Black Pearl,Red,5.0,4,Gas,12.2L/100km,9.0L - 9.5L/100km,43880
1,2018,Acura,MDX,77127 km,SUV,V6 Cylinder Engine,9 Speed Automatic,AWD,Modern Steel Metallic,Black,5.0,4,Gas,12.6L/100km,9.0L/100km,36486
2,2019,Acura,RDX,33032 km,SUV,2.0L 4cyl,10 Speed Automatic,AWD,White Diamond Pearl,Black,5.0,4,Premium Unleaded,11.0L/100km,8.6L/100km,40888
3,2020,Acura,RDX,50702 km,SUV,4 Cylinder Engine,Automatic,AWD,Platinum White Pearl,Black,5.0,4,Gas,11.0L/100km,8.6L/100km,44599
4,2021,Acura,RDX,67950 km,SUV,4 Cylinder Engine,Automatic,AWD,Apex Blue Pearl,Red,5.0,4,Gas,11.3L/100km,9.1L/100km,46989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24193,2017,Volvo,S90,81000 km,Sedan,4 Cylinder Engine,Automatic,AWD,White,Black,5.0,5 doors,Gasoline,10.8L/100km,8.7L/100km,34680
24194,2020,Volvo,XC40,92450 km,SUV,2.0,Automatic,AWD,Black,Black,5.0,5,Gas,10.8L/100km,8.7L/100km,35898
24195,2017,Volvo,XC90,92000 km,Hatchback,4 Cylinder Engine,Automatic,AWD,Grey,Black,5.0,4 doors,Gasoline,10.8L/100km,8.7L/100km,38000
24196,2018,Volvo,XC90,67000 km,SUV,4 Cylinder Engine,Automatic,AWD,Black,Black,5.0,4 doors,Gasoline,10.8L/100km,8.7L/100km,45000


In [11]:
df["engine_size"].unique()

<StringArray>
[                              'V6 Cylinder Engine',
                                        '2.0L 4cyl',
                                '4 Cylinder Engine',
                                             '3.5L',
                                        '3.5L 6cyl',
 'Intercooled Turbo Premium Unleaded I-4 2.0 L/122',
                          'V6 Cylinder Engine 3.5L',
                                       '6 Cylinder',
                           '2.4L 4 Cylinder Engine',
                                              '3.5',
 ...
                        '2.5L I5 150HP 170FT. LBS.',
                                     '2.5L L5 DOHC',
                                           '2.0 L.',
                                     '2.0L 150.0hp',
                                 '2.0L L4 SOHC TDI',
                                             '1.8T',
                                     '1.4L 170.0hp',
                                          '1.8 TSI',
                           

In [12]:
df["transmission"].unique()

<StringArray>
[                '9 Speed Automatic',                '10 Speed Automatic',
                         'Automatic',                    '6 Speed Manual',
                 '8 Speed Automatic',                 '6 Speed Automatic',
 '8 Speed Automatic with auto-shift',                 '5 Speed Automatic',
                    '5 Speed Manual',                 '4 Speed Automatic',
                            'Manual', '6 Speed Automatic with auto-shift',
 '7 Speed Automatic with auto-shift',                        'Sequential',
                 '7 Speed Automatic',                 '1 Speed Automatic',
                               'CVT',                    '4 Speed Manual',
                 '3 Speed Automatic',                    '7 Speed Manual',
 '5 Speed Automatic with auto-shift',                   'F1 Transmission']
Length: 22, dtype: string

In [13]:
df["drivetrain"].unique()

<StringArray>
['AWD', 'FWD', '4X4', 'RWD', '4x4', '4WD', '2WD']
Length: 7, dtype: string

In [14]:
pd.set_option("display.max_rows", 2000)  # or a higher number

In [15]:
print(df["exterior_colour"].unique().tolist())

['Majestic Black Pearl', 'Modern Steel Metallic', 'White Diamond Pearl', 'Platinum White Pearl', 'Apex Blue Pearl', 'Crystal Black Pearl', 'Silver', 'Bellanova White Pearl', 'Dark Blue', 'Lunar Silver Metallic', 'Black', 'Alabaster Silver Metallic', 'San Marino Red', 'Obsidian Blue Pearl', 'Blue', 'Dark Grey', 'Performance Red Pearl', 'Fathomless Black Pearl', 'White', 'Aspen White Pearl', 'Silver Moon Metallic', 'Grey', 'Still Night Blue Pearl', 'Beige', 'Nighthawk Black Pearl', 'Radiant Ruby Pearl', 'Satin Silver Metallic', 'Liquid Carbon Metallic', 'Graphite Luster Metallic', 'Crystal White Pearl', 'Berlina Black', 'Gunmetal Metallic', 'Not Specified', 'Bellanova White', 'Slate Silver Metallic', 'Formal Black', 'Burgundy', 'Kona Coffee Metallic', 'Taffeta White', 'Mayan Bronze Metallic', 'Palladium Silver Metallic', 'Palladlum Metallic', 'Gray', 'Royal Blue Pearl', 'Green', 'Gold', 'Premium White Pearl', 'Maroon', 'Red', 'Monza Red Metallic', 'Giallo Prototipo', 'Montecarlo Blue Met

In [16]:

df["interior_colour"].unique()

<StringArray>
[       'Red',      'Black',        'Tan',       'Grey',      'Brown',
      'Cream',      'Beige',  'Dark Grey', 'Light Grey',      'White',
   'Charcoal',       'Blue',   'Burgundy',     'Orange',      'Stone',
      'Taupe']
Length: 16, dtype: string

In [17]:

df["passengers"].unique()

<FloatingArray>
[5.0, 7.0, 4.0, 2.0, 6.0, 3.0, 8.0, 9.0, 12.0, 15.0]
Length: 10, dtype: Float64

In [18]:
df["doors"].unique()

<StringArray>
[                       '4',                        '2',
                        '5',                  '4 doors',
                  '5 doors',                  '2 doors',
                  '3 doors', 'Other/Donâ\x80\x99t Know',
                        '3',                      '2.0']
Length: 10, dtype: string

In [19]:
df["fuel_type"].unique()


<StringArray>
[                     'Gas',         'Premium Unleaded',
                 'Gasoline',          'Gasoline Hybrid',
                   'Diesel',      'Gas/Electric Hybrid',
                 'Electric',                 'Flexible',
            'Gasoline Fuel',                    'Other',
         'Regular Unleaded',        'Gasoline - Hybrid',
 'E85- Gasoline(Flex Fuel)', 'Other/Donâ\x80\x99t Know',
  'Gaseous Fuel Compatible']
Length: 15, dtype: string

In [20]:

df["city"].unique()


<StringArray>
[        '12.2L/100km',         '12.6L/100km',         '11.0L/100km',
         '11.3L/100km',         '11.4L/100km',         '10.5L/100km',
          '9.6L/100km',         '11.2L/100km',         '12.4L/100km',
         '12.3L/100km',
 ...
 '14.0L - 15.0L/100km', '13.5L - 14.6L/100km',   '7.3L - 8.5L/100km',
   '8.2L - 8.3L/100km', '12.4L - 13.0L/100km',   '8.0L - 9.0L/100km',
  '9.5L - 10.9L/100km',  '9.3L - 10.1L/100km', '10.8L - 11.0L/100km',
  '9.6L - 10.1L/100km']
Length: 523, dtype: string

In [21]:

df["highway"].unique()


<StringArray>
[ '9.0L - 9.5L/100km',         '9.0L/100km',         '8.6L/100km',
         '9.1L/100km',         '9.4L/100km',         '7.7L/100km',
         '7.0L/100km',         '6.6L/100km',         '7.5L/100km',
         '9.2L/100km',
 ...
  '5.7L - 6.5L/100km',  '4.9L - 5.5L/100km', '9.2L - 10.0L/100km',
  '6.9L - 7.3L/100km',  '5.8L - 6.3L/100km',  '4.4L - 4.9L/100km',
  '7.1L - 7.5L/100km',  '7.0L - 7.1L/100km',  '6.9L - 7.4L/100km',
  '6.8L - 7.1L/100km']
Length: 418, dtype: string

In [22]:
import re

def parse_fuel_value(value):
    if pd.isna(value):
        return None

    # Extract all numeric values (handles ranges and single values)
    numbers = re.findall(r"\d+\.\d+", value)
    if not numbers:
        return None

    # Convert to float and average if it's a range
    numbers = [float(n) for n in numbers]
    return sum(numbers) / len(numbers)

df["city_consumption"] = df["city"].apply(parse_fuel_value)
df["highway_consumption"] = df["highway"].apply(parse_fuel_value)



In [23]:
def parse_drivetrain_values(value):
    if value == '4x4' or value == '4X4':
       value =  '4X4' 
    if value in ['AWD', '4WD', '4x4',]:
        value   =  '4X4'
# PZ 2WD zu FWD
    if value in ['2WD', 'FWD']:
        value   =  'FWD'
    return value

df["drivetrain"] = df["drivetrain"].apply(parse_drivetrain_values)   

In [24]:
df["drivetrain"].unique()

array(['4X4', 'FWD', 'RWD'], dtype=object)

In [25]:
df = df.drop(columns= ["city", "highway"])

In [26]:
df["mileage"] = (
    df["mileage"]
    .astype(str)
    .str.replace("km", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]  # Extract numeric part safely
    .astype(float)
)


In [27]:

df

,year_of_manufacture,manufacturer,model,mileage,body_type,engine_size,transmission,drivetrain,exterior_colour,interior_colour,passengers,doors,fuel_type,price,city_consumption,highway_consumption
0,2019,Acura,MDX,53052.0,SUV,V6 Cylinder Engine,9 Speed Automatic,4X4,Majestic Black Pearl,Red,5.0,4,Gas,43880,12.2,9.25
1,2018,Acura,MDX,77127.0,SUV,V6 Cylinder Engine,9 Speed Automatic,4X4,Modern Steel Metallic,Black,5.0,4,Gas,36486,12.6,9.00
2,2019,Acura,RDX,33032.0,SUV,2.0L 4cyl,10 Speed Automatic,4X4,White Diamond Pearl,Black,5.0,4,Premium Unleaded,40888,11.0,8.60
3,2020,Acura,RDX,50702.0,SUV,4 Cylinder Engine,Automatic,4X4,Platinum White Pearl,Black,5.0,4,Gas,44599,11.0,8.60
4,2021,Acura,RDX,67950.0,SUV,4 Cylinder Engine,Automatic,4X4,Apex Blue Pearl,Red,5.0,4,Gas,46989,11.3,9.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24193,2017,Volvo,S90,81000.0,Sedan,4 Cylinder Engine,Automatic,4X4,White,Black,5.0,5 doors,Gasoline,34680,10.8,8.70
24194,2020,Volvo,XC40,92450.0,SUV,2.0,Automatic,4X4,Black,Black,5.0,5,Gas,35898,10.8,8.70
24195,2017,Volvo,XC90,92000.0,Hatchback,4 Cylinder Engine,Automatic,4X4,Grey,Black,5.0,4 doors,Gasoline,38000,10.8,8.70
24196,2018,Volvo,XC90,67000.0,SUV,4 Cylinder Engine,Automatic,4X4,Black,Black,5.0,4 doors,Gasoline,45000,10.8,8.70


In [28]:
def clean_doors(value):
    try:
        value = str(value).lower()
        value = value.replace("doors", "").replace("door", "").strip()
        num = pd.to_numeric(value, errors="coerce")
        return int(num) if pd.notnull(num) else None
    except:
        return None

df["doors"] = df["doors"].apply(clean_doors)
# PZ 5 Türen zu 4 Türen
df["doors"] = df["doors"].replace(5, 4)
most_common_doors = df["doors"].mode().iloc[0]
df["doors"] = df["doors"].fillna(most_common_doors).astype(int)


In [29]:
def clean_passengers(value):
    try:
        num = pd.to_numeric(value, errors="coerce")
        return int(num) if pd.notnull(num) else None
    except:
        return None

df["passengers"] = df["passengers"].apply(clean_passengers)

# Fill missing with most frequent value
most_common_passengers = df["passengers"].mode().iloc[0]
df["passengers"] = df["passengers"].fillna(most_common_passengers)


In [30]:
def simplify_color(color_name):
    """
    Simplifies a detailed car color name into a more general, useful category.
    Prioritizes specific colors over general shades.
    """
    if pd.isna(color_name):
        return "other"
    
    # Normalize the color name to lowercase for consistent matching
    s = str(color_name).lower()

    # --- Primary and Secondary Colors ---
    # These are checked first to give them priority.
    if "red" in s or "rosso" in s or "crimson" in s or "ruby" in s or "scarlet" in s or "carmine" in s:
        return "red"
    if "blue" in s or "blu" in s or "navy" in s or "aqua" in s or "teal" in s:
        return "blue"
    if "green" in s or "verde" in s or "lime" in s:
        return "green"
    if "yellow" in s or "giallo" in s or "gold" in s: # Gold is a shade of yellow
        return "yellow"
    if "orange" in s or "arancio" in s or "copper" in s: # Copper is a shade of orange
        return "orange"
    if "purple" in s or "violet" in s or "mauve" in s or "plum" in s:
        return "purple"
    if "pink" in s:
        return "pink"
        
    # --- Achromatic Colors (Black, White, Gray, Silver) ---
    # These are the most common and need to be distinct.
    if "black" in s or "ebony" in s or "onyx" in s or "nero" in s or "noir" in s:
        return "black"
    if "white" in s or "ivory" in s or "bianco" in s or "alabaster" in s:
        return "white"
    if "grey" in s or "gray" in s or "graphite" in s or "charcoal" in s or "grigio" in s:
        return "gray"
    if "silver" in s or "platinum" in s or "steel" in s:
        return "silver"
        
    # --- Earth Tones ---
    if "brown" in s or "bronze" in s or "mocha" in s or "tan" in s or "beige" in s or "sand" in s:
        return "brown"

    # --- Metallic/Pearl as a fallback category ---
    # If no specific color was found, check for a metallic finish.
    if "metallic" in s or "pearl" in s or "chrome" in s:
        return "metallic"

    # --- Default Category ---
    # If none of the above keywords match.
    return "other"

df["exterior_colour"] = df["exterior_colour"].apply(simplify_color)

# PZ interior colour gruppiert


def simplify_interior_color(color_name):
    if pd.isna(color_name):
        return "other"
    s = str(color_name).lower()
    if "red" in s or "burgundy" in s:
        return "red"
    if "blue" in s:
        return "blue"
    if "black" in s:
        return "black"
    if "brown" in s or "tan" in s:
        return "brown"
    if "grey" in s or "gray" in s or "charcoal" in s:
        return "gray"
    if "white" in s:
        return "white"
    if "beige" in s or "cream" in s or "stone" in s or "taupe" in s:
        return "beige"
    if "orange" in s:
        return "orange"
    return "other"

df["interior_colour"] = df["interior_colour"].apply(simplify_interior_color)



In [31]:
df

,year_of_manufacture,manufacturer,model,mileage,body_type,engine_size,transmission,drivetrain,exterior_colour,interior_colour,passengers,doors,fuel_type,price,city_consumption,highway_consumption
0,2019,Acura,MDX,53052.0,SUV,V6 Cylinder Engine,9 Speed Automatic,4X4,black,red,5,4,Gas,43880,12.2,9.25
1,2018,Acura,MDX,77127.0,SUV,V6 Cylinder Engine,9 Speed Automatic,4X4,silver,black,5,4,Gas,36486,12.6,9.00
2,2019,Acura,RDX,33032.0,SUV,2.0L 4cyl,10 Speed Automatic,4X4,white,black,5,4,Premium Unleaded,40888,11.0,8.60
3,2020,Acura,RDX,50702.0,SUV,4 Cylinder Engine,Automatic,4X4,white,black,5,4,Gas,44599,11.0,8.60
4,2021,Acura,RDX,67950.0,SUV,4 Cylinder Engine,Automatic,4X4,blue,red,5,4,Gas,46989,11.3,9.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24193,2017,Volvo,S90,81000.0,Sedan,4 Cylinder Engine,Automatic,4X4,white,black,5,4,Gasoline,34680,10.8,8.70
24194,2020,Volvo,XC40,92450.0,SUV,2.0,Automatic,4X4,black,black,5,4,Gas,35898,10.8,8.70
24195,2017,Volvo,XC90,92000.0,Hatchback,4 Cylinder Engine,Automatic,4X4,gray,black,5,4,Gasoline,38000,10.8,8.70
24196,2018,Volvo,XC90,67000.0,SUV,4 Cylinder Engine,Automatic,4X4,black,black,5,4,Gasoline,45000,10.8,8.70


In [32]:
# PZ flex-fuel und other durch gasoline ersetzt

def simplify_fuel_type(fuel):
    fuel = str(fuel).lower()
    if "hybrid" in fuel: return "hybrid"
    elif any(f in fuel for f in ["gas", "gasoline", "unleaded"]): return "gasoline"
    elif "diesel" in fuel: return "diesel"
    elif "electric" in fuel: return "electric"
    elif any(f in fuel for f in ["flex", "e85", "flexible"]): return "gasoline"
    else: return "gasoline"

    
df["fuel_type"] = df["fuel_type"].apply(simplify_fuel_type)
 


In [33]:
def simplify_transmission(transmission_name):
    """Simplifies the transmission name into a smaller set of categories."""
    if pd.isna(transmission_name):
        return 'Other'
        
    s = str(transmission_name).lower()

    if 'manual' in s:
        return 'Manual'
    if 'cvt' in s:
        return 'CVT'
    # 'F1' or 'sequential' are specific high-performance types
    if 'f1' in s or 'sequential' in s:
        return 'High-Performance'
    # If none of the above, it's an automatic
    if 'automatic' in s:
        return 'Automatic'
        
    # Fallback for any other unknown types
    return 'Other'

# --- How to apply it to your DataFrame ---
# df['transmission_simplified'] = df['transmission'].apply(simplify_transmission)

# --- Check the result ---
# print(df['transmission_simplified'].value_counts())

df["transmission"] = df["transmission"].apply(simplify_transmission)

df["transmission"].unique()

array(['Automatic', 'Manual', 'High-Performance', 'CVT'], dtype=object)

In [34]:
print(df["engine_size"].unique().tolist())


import re

def simplify_engine_features(engine_string):
    """
    Extracts displacement, cylinders, and engine type from a complex string.
    Returns a tuple: (displacement_litres, num_cylinders, engine_type).
    """
    if pd.isna(engine_string):
        return (np.nan, np.nan, 'Other')

    s = str(engine_string).lower()

    # --- Initialize default values ---
    displacement = np.nan
    cylinders = np.nan
    engine_type = 'Other'

    # --- Feature Extraction using Regular Expressions ---

    # 1. Extract Displacement (in Litres)
    # Looks for patterns like "3.5l", "2.0 l", "4.4", "2l"
    displacement_match = re.search(r'(\d\.\d+|\d+)\s*l', s)
    if not displacement_match:
        # Fallback for patterns like "3.5" without an 'l'
        displacement_match = re.search(r'(\d\.\d+)', s)
    
    if displacement_match:
        try:
            val = float(displacement_match.group(1))
            # Plausibility check: displacement is rarely over 10L for cars
            if 0.6 <= val <= 10.0:
                displacement = val
            # Handle cases where a year might be accidentally captured (e.g. 201HP)
            elif val > 100 and len(str(int(val))) == 4:
                 pass # This is likely a year or HP rating, so ignore
            elif val > 10 and val < 100:
                displacement = val / 10 # Corrects for typos like '35' instead of '3.5'

        except (ValueError, IndexError):
            pass

    # 2. Extract Number of Cylinders
    # Looks for "v6", "4cyl", "8 cylinder", "i-4"
    cyl_patterns = [
        r'(\d+)\s*c(?:yl|ylinder)', 
        r'v-?(\d+)', 
        r'i-?(\d+)',
        r'w(\d+)',
        r'(\d+)\s*cyl'
    ]
    for pattern in cyl_patterns:
        cyl_match = re.search(pattern, s)
        if cyl_match:
            try:
                cylinders = int(cyl_match.group(1))
                break # Stop after the first successful match
            except (ValueError, IndexError):
                continue
                
    # 3. Determine Engine Type (Configuration)
    if "electric" in s or "motor" in s:
        engine_type = 'Electric'
        cylinders = 0  # Electric motors have 0 cylinders
        displacement = 0.0
    elif "hybrid" in s:
        engine_type = 'Hybrid'
    elif "v" in s:
        engine_type = 'V-Engine'
    elif "i-" in s or "inline" in s or "straight" in s:
        engine_type = 'Inline'
    elif "w12" in s or "w16" in s:
        engine_type = 'W-Engine'
    elif "boxer" in s or "h-4" in s or "h-6" in s:
        engine_type = 'Boxer'
    elif "rotary" in s:
        engine_type = 'Rotary'
    elif cylinders is not np.nan:
        # If we found cylinders but no type, assume Inline as it's common
        engine_type = 'Inline'

    return (displacement, cylinders, engine_type)


# --- HOW TO USE IT ---
# Assume `df` is your DataFrame

# Apply the function to the 'engine_size' column. It returns a list of tuples.
engine_features = df['engine_size'].apply(simplify_engine_features)

# Create new columns in your DataFrame from the list of tuples.
df[['engine_displacement_L', 'engine_cylinders', 'engine_type']] = pd.DataFrame(engine_features.tolist(), index=df.index)

# Now, you can drop the original messy column
df.drop('engine_size', axis=1, inplace=True)

# You would then proceed to impute (fill) any remaining NaN values in the new numeric columns.
# For example:
df['engine_displacement_L'].fillna(df['engine_displacement_L'].median(), inplace=True)
df['engine_cylinders'].fillna(df['engine_cylinders'].median(), inplace=True)

['V6 Cylinder Engine', '2.0L 4cyl', '4 Cylinder Engine', '3.5L', '3.5L 6cyl', 'Intercooled Turbo Premium Unleaded I-4 2.0 L/122', 'V6 Cylinder Engine 3.5L', '6 Cylinder', '2.4L 4 Cylinder Engine', '3.5', '4 Cylinder Engine 2.4L', 'Premium Unleaded I-4 2.4 L/144', 'V-6 cyl', '3.5L V6 Cylinder Engine', 'I-4 cyl', '6', '2.0L 4 Cylinder Engine', '4 Cylinder Engine 2.0L', 'Premium Unleaded V-6 3.5 L/212', 'V6 Cylinder Engine 3.0L', '2.4L L4 DOHC 16V', '3.5L V6 F SOHC 24V', '2.4', '3.7', '3.7L 6cyl', '2.0', '1.7L L4 SOHC 16V', '4 Cylinder Engine 2.3L', '2.4L 4cyl', '4 CYL', '201HP 2.4L 4 Cylinder Engine', '3.5L SOHC', '2.5', '3.5 V6 Hybrid', '3.5L SOHC 24-Valve VTEC V6 -inc: Aluminum-alloy', '2.5L', '2.4L', '3.2L V6 258HP 233FT. LBS.', '2.3', '3.2L', '1.7L 4cyl', '2.9L 6cyl', 'I4', '4 cylinder engine', 'Four-cylinder DOHC engine', '2', '2.2L L4 DOHC 16V', '4 Cylinder', '2L TURBO 4cyl.', '3L TURBO 6cyl.', '2.0 TFSI 4-Cyl 220hp Engine', '2.0L L4 DOHC 16V', '8 Cylinder Engine', '3.0 Liter TFSI'

C:\Users\Anwender\AppData\Local\Temp\ipykernel_15064\2891161838.py:101: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['engine_displacement_L'].fillna(df['engine_displacement_L'].median(), inplace=True)
C:\Users\Anwender\AppData\Local\Temp\ipykernel_15064\2891161838.py:102: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are s

In [35]:
df_test = df[df["price"]>=3000000]

In [36]:
df_test

,year_of_manufacture,manufacturer,model,mileage,body_type,transmission,drivetrain,exterior_colour,interior_colour,passengers,doors,fuel_type,price,city_consumption,highway_consumption,engine_displacement_L,engine_cylinders,engine_type
3885,2019,Bugatti,Unlisted,1226.0,SUV,Automatic,4X4,blue,black,2,2,gasoline,3999998,10.8,8.7,2.4,16.0,W-Engine
21076,2020,Lamborghini,Unlisted,343.0,Coupe,Automatic,4X4,red,black,2,2,gasoline,3699998,10.8,8.7,2.4,12.0,Inline


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24198 entries, 0 to 24197
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   year_of_manufacture    24198 non-null  Int64  
 1   manufacturer           24198 non-null  string 
 2   model                  24198 non-null  string 
 3   mileage                24198 non-null  float64
 4   body_type              24198 non-null  string 
 5   transmission           24198 non-null  object 
 6   drivetrain             24198 non-null  object 
 7   exterior_colour        24198 non-null  object 
 8   interior_colour        24198 non-null  object 
 9   passengers             24198 non-null  int64  
 10  doors                  24198 non-null  int32  
 11  fuel_type              24198 non-null  object 
 12  price                  24198 non-null  Int64  
 13  city_consumption       24198 non-null  float64
 14  highway_consumption    24198 non-null  float64
 15  en

In [38]:
import pandas as pd
import json
import numpy as np
import webcolors



print("--- Generating UI configuration file ---")

ui_config = {}

# --- STEP 1: Define the color mapping logic ---
# A fallback map for common names not in the standard CSS list
# PZ beige hinzugefuegt (auskommentiert, da in webcolors enthalten)
color_fallback_map = {
    "black": "#000000",
    "white": "#FFFFFF",
    "silver": "#C0C0C0",
    "gray": "#808080",
    #"beige": "#F5F5DC",
    "red": "#D9534F",
    "blue": "#007bff",
    "green": "#5CB85C",
    "yellow": "#F0AD4E",
    "brown": "#A0522D",
    "orange": "#ED9C28",
    "purple": "#800080",
    "pink": "#E75480",
    "other": "#777777",
    "metallic": "#AAAAAA"
}

def get_color_code(color_name):
    """Converts a color name to a hex code."""
    try:
        # First, try the webcolors library
        return webcolors.name_to_hex(color_name)
    except ValueError:
        # If it fails, try our fallback map
        return color_fallback_map.get(color_name.lower(), "#777777") # Default to gray if not found

# --- STEP 2: Get unique values for all categorical columns ---
categorical_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
print(f"Found categorical columns: {categorical_cols}")

for col in categorical_cols:
    unique_values = sorted(df[col].unique().tolist())
    ui_config[col] = unique_values

# --- STEP 3: Create the special manufacturer-to-model mapping ---
print("Creating manufacturer-to-model mapping...")
manufacturer_models_mapping = df.groupby('manufacturer')['model'].unique().apply(list).to_dict()
ui_config['manufacturer_models'] = manufacturer_models_mapping

# --- STEP 4: Create the new 'color_map' entry ---
print("Creating color map...")
all_colors = sorted(list(set(df['exterior_colour'].unique()) | set(df['interior_colour'].unique())))
color_map = {name: get_color_code(name) for name in all_colors}
ui_config['color_map'] = color_map

# --- STEP 5: Write the final dictionary to a JSON file ---
output_filename = 'ui_config.json'
with open(output_filename, 'w') as f:
    json.dump(ui_config, f, indent=4)


print(f"\n✅ Success! Configuration with color map saved to '{output_filename}'.")

--- Generating UI configuration file ---
Found categorical columns: ['manufacturer', 'model', 'body_type', 'transmission', 'drivetrain', 'exterior_colour', 'interior_colour', 'fuel_type', 'engine_type']
Creating manufacturer-to-model mapping...
Creating color map...

✅ Success! Configuration with color map saved to 'ui_config.json'.


In [39]:

def simplify_body_type(body_type_name):
    """Simplifies the body_type name into a smaller set of standard categories."""
    if pd.isna(body_type_name):
        return 'Other'
        
    s = str(body_type_name).lower()

    # The most important and specific check first: Trucks
    if 'truck' in s or 'cab' in s or 'crew' in s:
        return 'Truck'
    
    # Common car types
    if 'suv' in s:
        return 'SUV'
    if 'sedan' in s:
        return 'Sedan'
    if 'hatchback' in s:
        return 'Hatchback'
    
    # Groupings for similar styles
    if 'coupe' in s or 'roadster' in s:
        return 'Coupe'
    if 'wagon' in s:
        return 'Wagon'
    if 'convertible' in s or 'cabriolet' in s:
        return 'Convertible'
    if 'van' in s:
        return 'Van'
    
    # Standalone categories
    if 'compact' in s:
        return 'Compact'
        
    # Fallback for any other unknown types
    return 'Other'

# --- How to apply it to your DataFrame ---
df['body_type'] = df['body_type'].apply(simplify_body_type)

# --- Check the result ---
# print("Original vs. Simplified Body Types:")
# print(df[['body_type', 'body_type_simplified']].head(10))
# print("\nNew Category Counts:")
# print(df['body_type_simplified'].value_counts())

In [40]:
df["body_type"].unique()


array(['SUV', 'Sedan', 'Coupe', 'Hatchback', 'Wagon', 'Convertible',
       'Truck', 'Compact', 'Van'], dtype=object)

In [41]:
df["engine_type"].isnull().any()


False

In [42]:
# PZ data cleaning: 
# Unlisted, viele km, alte Baujahre loeschen
# e-autos Verbauch und Motor = 0 setzen

df = df.drop(df.loc[df["model"] == "Unlisted"].index)
df = df.drop(df.loc[df["mileage"] > 1000000].index)
df = df.drop(df.loc[df["year_of_manufacture"] < 1950].index)
df.loc[df["fuel_type"] == "electric", ["city_consumption", "highway_consuption", "engine_displacement", "engine_cylinders"]] = 0
df.loc[df["fuel_type"] == "electric", "engine_type"] = "Electric"



In [45]:
df = df.drop(["highway_consuption", "engine_displacement"], axis=1)

In [58]:
import numpy as np
from datetime import datetime

def augment_data_with_car_state(df: pd.DataFrame) -> pd.DataFrame:
    """
    Augments a car sales DataFrame with a probabilistically inferred 'car_state'
    (in 4 classes) and adjusts the 'price' based on that state.

    The 4 car state classes are:
    1: Very Good State (Minimal wear, like new)
    2: Minor Damage (e.g., light cosmetic scratches)
    3: Moderate Damage (e.g., visible dents, significant paintwork needed)
    4: Severe Damage (e.g., major bodywork, structural issues)

    Args:
        df (pd.DataFrame): The input DataFrame. Must contain the columns:
                           'year_of_manufacture', 'mileage', and 'price'.

    Returns:
        pd.DataFrame: A new DataFrame with an added 'car_state' column and an
                      adjusted 'price' column.
    """
    
    # --- 1. Configuration & Tunable Parameters ---
    
    CURRENT_YEAR = datetime.now().year
    BENCHMARK_MILEAGE_PER_YEAR = 15000 
    
    AGE_WEIGHT = 1.0
    MILEAGE_WEIGHT = 1.0
    OWNERSHIP_LUCK_STD_DEV = 0.5 
    
    # --- NEW: State Thresholds for 4 Classes ---
    # We now only need 3 thresholds to define the 4 groups.
    STATE_THRESHOLDS = {
        1: 0.5,   # Scores below 0.5 are "Very Good"
        2: 1.5,   # Scores between 0.5 and 1.5 are "Minor"
        3: 2.5,   # Scores between 1.5 and 2.5 are "Moderate"
                  # Scores above 2.5 are "Severe"
    }

    # --- NEW: Damage Cost Parameters for 4 Classes ---
    DAMAGE_COSTS = {
        1: lambda p: 0,                         # No cost for a very good car
        2: lambda p: 500,                       # Flat rate for minor cosmetic repairs
        3: lambda p: p * 0.15 + 1000,           # Cost for moderate damage (e.g., new panel)
        4: lambda p: p * 0.30 + 2500,           # Cost for severe damage (structural/multiple parts)
    }

    # --- 2. Create a Working Copy ---
    df_augmented = df.copy()

    # --- 3. Calculate Condition Score Components (Unchanged) ---
    
    df_augmented['age'] = CURRENT_YEAR - df_augmented['year_of_manufacture']
    df_augmented['age_for_rate_calc'] = df_augmented['age'].clip(lower=1.0)
    
    age_score = (df_augmented['age'] / 10.0) * AGE_WEIGHT
    
    actual_mileage_per_year = df_augmented['mileage'] / df_augmented['age_for_rate_calc']
    mileage_rate_deviation = actual_mileage_per_year - BENCHMARK_MILEAGE_PER_YEAR
    mileage_score = (mileage_rate_deviation / 10000.0) * MILEAGE_WEIGHT
    
    num_rows = len(df_augmented)
    ownership_luck = np.random.normal(loc=0, scale=OWNERSHIP_LUCK_STD_DEV, size=num_rows)
    
    # --- 4. Calculate Final Score and Assign State (Adjusted for 4 Classes) ---
    
    df_augmented['condition_score'] = age_score + mileage_score + ownership_luck
    
    # Define the conditions and choices for the 4 new states
    conditions = [
        (df_augmented['condition_score'] < STATE_THRESHOLDS[1]),
        (df_augmented['condition_score'] < STATE_THRESHOLDS[2]),
        (df_augmented['condition_score'] < STATE_THRESHOLDS[3]),
        (df_augmented['condition_score'] >= STATE_THRESHOLDS[3]) # Everything else is severe
    ]
    choices = [1, 2, 3, 4]
    
    # Use np.select to assign the state; the default is now 4 (Severe)
    df_augmented['car_state'] = np.select(conditions, choices, default=4)
    
    # --- 5. Adjust Price Based on the New State ---
    
    df_augmented['damage_cost'] = df_augmented.apply(
        lambda row: DAMAGE_COSTS[row['car_state']](row['price']),
        axis=1
    )
    
    df_augmented['price'] = (df_augmented['price'] - df_augmented['damage_cost']).clip(lower=500)
    
    # --- 6. Final Cleanup ---
    df_final = df_augmented.drop(columns=['age', 'age_for_rate_calc', 'condition_score', 'damage_cost'])
    
    df_final['car_state'] = df_final['car_state'].astype(int)

    print("Data augmentation complete with 4 damage classes.")
    print("Distribution of generated car states:")
    print(df_final['car_state'].value_counts(normalize=True).sort_index())
    
    return df_final

In [59]:
df = augment_data_with_car_state(df)

Data augmentation complete with 4 damage classes.
Distribution of generated car states:
car_state
1    0.658259
2    0.251313
3    0.079347
4    0.011081
Name: proportion, dtype: float64


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24185 entries, 0 to 24197
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   year_of_manufacture    24185 non-null  Int64  
 1   manufacturer           24185 non-null  string 
 2   model                  24185 non-null  string 
 3   mileage                24185 non-null  float64
 4   body_type              24185 non-null  object 
 5   transmission           24185 non-null  object 
 6   drivetrain             24185 non-null  object 
 7   exterior_colour        24185 non-null  object 
 8   interior_colour        24185 non-null  object 
 9   passengers             24185 non-null  int64  
 10  doors                  24185 non-null  int32  
 11  fuel_type              24185 non-null  object 
 12  price                  24185 non-null  Float64
 13  city_consumption       24185 non-null  float64
 14  highway_consumption    24185 non-null  float64
 15  engine_

In [61]:
from sklearn.model_selection import train_test_split

# Shuffle and split
train_df, test_df = train_test_split(df, test_size=0.05, random_state=42, shuffle=True)

# Save to CSV
# PZ Pfad anpassen!!!
train_df.to_csv(r"detailed_car_sales_data_train.csv", index=False)
test_df.to_csv(r"detailed_car_sales_data_test.csv", index=False)

df.to_csv("detailed_car_sales_data_all.csv", index=False)

In [63]:
df[df["car_state"]==4]

,year_of_manufacture,manufacturer,model,mileage,body_type,transmission,drivetrain,exterior_colour,interior_colour,passengers,doors,fuel_type,price,city_consumption,highway_consumption,engine_displacement_L,engine_cylinders,engine_type,car_state
359,2003,Acura,EL,240500.0,Sedan,Automatic,FWD,silver,black,5,4,gasoline,500.0,10.80,8.70,2.40,4.0,Inline,4
371,2005,Acura,MDX,270000.0,SUV,Automatic,4X4,black,black,5,4,gasoline,500.0,10.80,8.70,2.40,4.0,Inline,4
391,2001,Acura,TL,225000.0,Sedan,Automatic,FWD,yellow,black,5,4,gasoline,2153.74,10.80,8.70,2.40,4.0,Inline,4
550,1973,Alfa Romeo,Romeo,70917.0,SUV,Automatic,4X4,yellow,black,5,4,gasoline,16352.96,10.80,8.70,2.40,4.0,Inline,4
866,2014,Audi,allroad,146965.0,Sedan,Automatic,4X4,silver,beige,5,4,gasoline,8423.5,10.80,8.70,2.00,4.0,Other,4
2238,2006,Volvo,XC90,357000.0,Wagon,Automatic,4X4,black,black,5,4,gasoline,500.0,10.80,8.70,2.40,4.0,Inline,4
2586,1987,BMW,325I,234345.0,Sedan,Automatic,RWD,blue,black,5,4,gasoline,500.0,10.80,8.70,2.40,4.0,Inline,4
2945,1973,BMW,2002,58500.0,Coupe,Manual,RWD,green,brown,4,2,gasoline,16306.34,10.80,8.70,2.40,4.0,Inline,4
2946,1987,BMW,325I,234345.0,Sedan,Automatic,RWD,blue,black,5,4,gasoline,500.0,10.80,8.70,2.40,4.0,Inline,4
3005,1997,BMW,Z3,148187.0,Convertible,Manual,RWD,green,brown,2,2,gasoline,9671.6,10.20,7.60,2.40,4.0,Inline,4


In [48]:
# --- User Configuration ---
# Create a sample DataFrame.
# IMPORTANT: Replace this with your actual DataFrame containing car manufacturers and models.
data = {'manufacturer': (df['manufacturer'].unique()).tolist(),
        'model': (df['model'].unique()).tolist()
        }
        
        

In [65]:
!pip install tensorflow_datasets

  Obtaining dependency information for tensorflow_datasets from https://files.pythonhosted.org/packages/16/e0/657192dbc03636532ccbd5c90669d31a65187365b99ba685db36bb31dd67/tensorflow_datasets-4.9.9-py3-none-any.whl.metadata
  Obtaining dependency information for dm-tree from https://files.pythonhosted.org/packages/35/3e/a46933e0157b0ac87619a754ce1a796b2afc6386fca7c11f95c010f40745/dm_tree-0.1.9-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for etils[edc,enp,epath,epy,etree]>=1.9.1 from https://files.pythonhosted.org/packages/e7/98/87b5946356095738cb90a6df7b35ff69ac5750f6e783d5fbcc5cb3b6cbd7/etils-1.13.0-py3-none-any.whl.metadata
  Obtaining dependency information for immutabledict from https://files.pythonhosted.org/packages/63/7b/04ab6afa1ff7eb9ccb09049918c0407b205f5009092c0416147d163e4e2b/immutabledict-4.2.2-py3-none-any.whl.metadata
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency infor


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [69]:
!pip install tensorflow

  Obtaining dependency information for tensorflow from https://files.pythonhosted.org/packages/e3/f8/9246d3c7e185a29d7359d8b12b3d70bf2c3150ecf1427ec1382290e71a56/tensorflow-2.20.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for astunparse>=1.6.0 from https://files.pythonhosted.org/packages/2b/03/13dde6512ad7b4557eb792fbcf0c653af6076b81e5941d36ec61f7ce6028/astunparse-1.6.3-py2.py3-none-any.whl.metadata
  Obtaining dependency information for gast!=0.5.0,!=0.5.1,!=0.5.2,>=0.2.1 from https://files.pythonhosted.org/packages/a3/61/8001b38461d751cd1a0c3a6ae84346796a5758123f3ed97a1b121dfbf4f3/gast-0.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for google_pasta>=0.1.1 from https://files.pythonhosted.org/packages/a3/de/c648ef6835192e6e2cc03f40b19eeda4382c49b5bafb43d88b931c4c74ac/google_pasta-0.2.0-py3-none-any.whl.metadata
  Obtaining dependency information for libclang>=13.0.0 from https://files.pythonhosted.org/packages/0b/2d/3f480b1e1d31eb3d6de5e3


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [72]:
!kaggle datasets download -d jutrera/stanford-car-dataset-by-classes-folder -p C:/Users/Anwender/Downloads/Master/data_ui/cars196

... resuming from 1764753408 bytes (195471965 bytes left) ...




 90%|█████████ | 1.64G/1.83G [00:00<?, ?B/s]
 90%|█████████ | 1.64G/1.83G [00:00<02:13, 1.46MB/s]
 90%|█████████ | 1.65G/1.83G [00:00<01:18, 2.46MB/s]
 90%|█████████ | 1.65G/1.83G [00:01<01:00, 3.15MB/s]
 90%|█████████ | 1.65G/1.83G [00:01<00:52, 3.64MB/s]
 90%|█████████ | 1.65G/1.83G [00:01<00:49, 3.84MB/s]
 90%|█████████ | 1.65G/1.83G [00:01<00:48, 3.94MB/s]
 90%|█████████ | 1.65G/1.83G [00:02<00:47, 4.00MB/s]
 90%|█████████ | 1.65G/1.83G [00:02<00:46, 4.01MB/s]
 91%|█████████ | 1.65G/1.83G [00:02<00:54, 3.44MB/s]
 91%|█████████ | 1.65G/1.83G [00:03<00:55, 3.36MB/s]
 91%|█████████ | 1.65G/1.83G [00:03<00:54, 3.40MB/s]
 91%|█████████ | 1.66G/1.83G [00:03<00:54, 3.35MB/s]
 91%|█████████ | 1.66G/1.83G [00:04<00:54, 3.31MB/s]
 91%|█████████ | 1.66G/1.83G [00:04<00:53, 3.40MB/s]
 91%|█████████ | 1.66G/1.83G [00:04<00:52, 3.42MB/s]
 91%|█████████ | 1.66G/1.83G [00:04<00:52, 3.44MB/s]
 91%|█████████ | 1.66G/1.83G [00:05<00:50, 3.52MB/s]
 91%|█████████ | 1.66G/1.83G [00:05<00:48, 3.62MB/s]


In [9]:
!pip install bing_image_downloader


  Obtaining dependency information for bing_image_downloader from https://files.pythonhosted.org/packages/24/bf/105be56c70c4141a3d42dbb9cdd30dfbd3bb3fc135879fda21d1bfa1eb51/bing_image_downloader-1.1.2-py3-none-any.whl.metadata



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
data = df[["manufacturer", "model"]]

In [2]:
!pip install webdriver-manager

  Obtaining dependency information for webdriver-manager from https://files.pythonhosted.org/packages/b5/b5/3bd0b038d80950ec13e6a2c8d03ed8354867dc60064b172f2f4ffac8afbe/webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [74]:
import os
import shutil
import random

def copy_sample_images(src_root, dst_root, n=3):
    """
    Iterate through each subfolder in src_root, copy n images from each
    into dst_root. Images are chosen randomly if more than n exist.
    
    Parameters:
        src_root (str): Path to the source folder containing subfolders.
        dst_root (str): Path to the destination folder.
        n (int): Number of images to copy per subfolder (default=3).
    """
    os.makedirs(dst_root, exist_ok=True)

    # Walk through subfolders
    for subdir in os.listdir(src_root):
        subdir_path = os.path.join(src_root, subdir)
        if os.path.isdir(subdir_path):
            # Collect image files (filter common extensions)
            images = [f for f in os.listdir(subdir_path) 
                      if f.lower().endswith((".jpg", ".jpeg", ".png"))]

            if not images:
                continue  # skip empty folders

            # Pick n images (random sample if more than n)
            selected = images if len(images) <= n else random.sample(images, n)

            # Copy to destination
            for img in selected:
                src_img = os.path.join(subdir_path, img)
                dst_img = os.path.join(dst_root, f"{subdir}_{img}")
                shutil.copy(src_img, dst_img)

    print(f"Copied up to {n} images from each subfolder into {dst_root}")


In [76]:
copy_sample_images(src_root = "C:/Users/Anwender/Downloads/Master/data_ui/cars196/stanford-car-dataset-by-classes-folder/car_data/car_data/test", dst_root="C:/Users/Anwender/Downloads/Master/data_ui/car_state/data3a/training" , n=3)

Copied up to 3 images from each subfolder into C:/Users/Anwender/Downloads/Master/data_ui/car_state/data3a/training


In [ ]:
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# below optional: run headless with stable window size only if needed
# options.add_argument("window-size=1200,900")
#
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--headless=new")



In [18]:
# Import packages
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

import json
import requests
import os
import io
import time
from PIL import Image
import base64

# --- Configuration ---
BASE_DIR = "C:\\Users\\Anwender\\Downloads\\Master\\data_ui\\detailed_images"
JSON_FILE_PATH = "C:\\Users\\Anwender\\Downloads\\Master\\data_ui\\manufacturer_models.json"
# --------------------

class GoogleScraper():
    def __init__(self, webdriver:webdriver):
        self.wd = webdriver

    def _build_query(self, query:str):
        from urllib.parse import quote
        query_encoded = quote(query)
        return f"https://www.google.com/search?safe=off&site=&tbm=isch&source=hp&q={query_encoded}&oq={query_encoded}&gs_l=img"


    def get_image_source(self, query: str):
        """
        Fetch high-res image source from Google Images robustly (handles staleness - 2025).
        """
        import requests
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
        from selenium.webdriver.support import expected_conditions as EC
        import time

        try:
            self.wd.get(self._build_query(query))
        except Exception as e:
            print(f"ERROR: Could not navigate to search page for '{query}'. {e}")
            return None

        # Cookie dialog
        try:
            consent_button = WebDriverWait(self.wd, 5).until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//button[contains(., 'Accept all') or contains(., 'Alle akzeptieren')]")
                )
            )
            consent_button.click()
            time.sleep(1)
        except Exception:
            pass

        try:
            WebDriverWait(self.wd, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "img"))
            )
        except Exception as e:
            print(f"ERROR: No images appeared for '{query}'. {e}")
            return None

        try:
            # 1. Find all visible thumbnails
            thumbnails = self.wd.find_elements(By.CSS_SELECTOR, "img.YQ4gaf") or \
                        self.wd.find_elements(By.CSS_SELECTOR, "img.rg_i")
            if not thumbnails:
                print(f"ERROR: No thumbnails found for '{query}'.")
                return None

            print(f"Found {len(thumbnails)} image thumbnails for '{query}'.")

            # 2. Click on the first thumbnail, re-query for preview images
            thumbnails[0].click()
            time.sleep(2.5)  # Give time for preview panel image to load

            # 3. After clicking, re-fetch all images in overlay/right panel
            highres_candidates = self.wd.find_elements(By.CSS_SELECTOR, "img")
            # Filter for .jpg, .jpeg, .png, .webp, .gif; prefer largest visible image src
            best_src = None
            best_size = 0
            for img in highres_candidates:
                src = img.get_attribute("src")
                if not src:
                    continue
                # Prefer only valid images, not icons or sprites
                if (src.startswith("http") or src.startswith("data:image")) and any(
                        ext in src for ext in [".jpg", ".jpeg", ".png", ".webp", ".gif"]):
                    try:
                        if src.startswith("http"):
                            resp = requests.head(src, timeout=5)
                            content_length = int(resp.headers.get("content-length", 0))
                        else:
                            content_length = len(src)
                        # largest size wins
                        if content_length > best_size:
                            best_size = content_length
                            best_src = src
                    except Exception:
                        continue

            if best_src:
                print(f"SUCCESS: High-res image source found ({best_size} bytes) for '{query}'.")
                return best_src

            print("ERROR: Could not detect suitable high-res image, falling back to thumbnail.")
            # 4. Fallback: Always re-fetch thumbnail to avoid staleness
            thumb = self.wd.find_elements(By.CSS_SELECTOR, "img.YQ4gaf") or \
                    self.wd.find_elements(By.CSS_SELECTOR, "img.rg_i")
            if thumb:
                thumb_src = thumb[0].get_attribute("src")
                return thumb_src if thumb_src else None
            return None

        except Exception as e:
            print(f"ERROR while trying to get high-resolution image for '{query}': {e}")
            return None




    def download_image(self, full_file_path:str, url_or_data:str):
        try:
            image_content = None
            if url_or_data.startswith('data:image'):
                header, encoded = url_or_data.split(',', 1)
                image_content = base64.b64decode(encoded)
            else:
                image_content = requests.get(url_or_data, timeout=10).content

            if not image_content:
                print("ERROR: Image content was empty.")
                return

            image_file = io.BytesIO(image_content)
            image = Image.open(image_file).convert('RGB')
            with open(full_file_path, 'wb') as f:
                image.save(f, "JPEG", quality=85)
            print(f"SUCCESS: Saved image to {full_file_path}")

        except Exception as e:
            print(f"ERROR: Could not download or save image. {e}")


# --- Main Driver Script ---
try:
    with open(JSON_FILE_PATH, 'r') as f:
        manufacturer_models = json.load(f)
    print(f"Successfully loaded {len(manufacturer_models)} manufacturers from JSON file.")
except FileNotFoundError:
    print(f"FATAL ERROR: Could not find JSON file at '{JSON_FILE_PATH}'")
    exit()

try:
    options = Options()
    options.add_argument("start-maximized") # Start browser maximized
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    
    service = Service(ChromeDriverManager().install())
    wd = webdriver.Chrome(service=service, options=options)
except Exception as e:
    print(f"FATAL ERROR: Could not start Selenium webdriver. Error: {e}")
    exit()

gs = GoogleScraper(wd)
if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)

try:
    for manufacturer, models in manufacturer_models.items():
        print(f"\n--- Processing Manufacturer: {manufacturer} ---")
        manufacturer_dir = os.path.join(BASE_DIR, manufacturer)
        if not os.path.exists(manufacturer_dir):
            os.makedirs(manufacturer_dir)

        for model in models:
            sanitized_model_name = model.replace('/', '_').replace('\\', '_')
            final_image_path = os.path.join(manufacturer_dir, f"{sanitized_model_name}.jpg")

            if os.path.exists(final_image_path):
                print(f"Skipping '{model}', image already exists.")
                continue

            query = f"{manufacturer} {sanitized_model_name} car"
            print(f"Processing model: {model} (Query: '{query}')")

            image_source = gs.get_image_source(query)

            if not image_source:
                print(f"-> FAILED: No image source found for '{query}'.")
                continue
            
            gs.download_image(full_file_path=final_image_path, url_or_data=image_source)
            time.sleep(1) # Add a small polite delay between requests
except Exception as e:
    import traceback
    print("ERROR: Exception during image extraction:")
    traceback.print_exc()

finally:
    print("\n✅ Image downloading process finished. Closing browser.")
    wd.quit()

Successfully loaded 51 manufacturers from JSON file.

--- Processing Manufacturer: Acura ---
Processing model: EL (Query: 'Acura EL car')
Found 217 image thumbnails for 'Acura EL car'.
ERROR: Could not detect suitable high-res image, falling back to thumbnail.
SUCCESS: Saved image to C:\Users\Anwender\Downloads\Master\data_ui\detailed_images\Acura\EL.jpg
Processing model: ILX (Query: 'Acura ILX car')
Found 222 image thumbnails for 'Acura ILX car'.
ERROR: Could not detect suitable high-res image, falling back to thumbnail.
SUCCESS: Saved image to C:\Users\Anwender\Downloads\Master\data_ui\detailed_images\Acura\ILX.jpg
Processing model: CSX (Query: 'Acura CSX car')
Found 219 image thumbnails for 'Acura CSX car'.
ERROR: Could not detect suitable high-res image, falling back to thumbnail.
SUCCESS: Saved image to C:\Users\Anwender\Downloads\Master\data_ui\detailed_images\Acura\CSX.jpg
Processing model: Integra (Query: 'Acura Integra car')
Found 225 image thumbnails for 'Acura Integra car'.


KeyboardInterrupt: 

In [8]:
# Import packages
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

import json
import requests
import os
import io
import time
from PIL import Image
import base64

# --- Configuration ---
BASE_DIR = "C:\\Users\\Anwender\\Downloads\\Master\\data_ui\\detailed_images"
JSON_FILE_PATH = "C:\\Users\\Anwender\\Downloads\\Master\\data_ui\\manufacturer_models.json"
# --------------------


def get_and_download_first_image(driver, query, save_path):
    """
    A simple, direct function to get the first image and download it.
    """
    try:
        # 1. Build and navigate to the search URL
        from urllib.parse import quote
        query_encoded = quote(query)
        url = f"https://www.google.com/search?safe=off&site=&tbm=isch&source=hp&q={query_encoded}&oq={query_encoded}&gs_l=img"
        driver.get(url)

        # 2. Handle the Cookie Consent Pop-up
        try:
            consent_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Accept all') or contains(., 'Alle akzeptieren')]"))
            )
            print("Cookie consent button found. Clicking it...")
            consent_button.click()
            time.sleep(1.5) # Wait for the page to stabilize after the click
        except Exception as e:
            print("No cookie consent button found, or it failed. Continuing...")

        # 3. Find the VERY FIRST image thumbnail and get its source
        # This is the most crucial step. We wait for it to be present and grab it immediately.
        thumbnail = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "img.rg_i"))
        )
        
        image_src = thumbnail.get_attribute('src')
        if not image_src:
            print("-> FAILED: First thumbnail had no source attribute.")
            return

        # 4. Download and save the image based on its source type
        image_content = None
        if image_src.startswith('data:image'):
            print("Source is Base64. Decoding...")
            header, encoded = image_src.split(',', 1)
            image_content = base64.b64decode(encoded)
        elif image_src.startswith('http'):
            print("Source is a URL. Downloading...")
            image_content = requests.get(image_src, timeout=10).content
        else:
            print(f"-> FAILED: Unknown image source format: {image_src[:50]}")
            return

        if not image_content:
            print("-> FAILED: Image content was empty after processing source.")
            return

        # 5. Save the image to the file
        image_file = io.BytesIO(image_content)
        image = Image.open(image_file).convert('RGB')
        with open(save_path, 'wb') as f:
            image.save(f, "JPEG", quality=85)
        print(f"-> SUCCESS: Saved image to {save_path}")

    except Exception as e:
        print(f"-> FAILED: An unexpected error occurred for query '{query}': {e}")


# --- Main Driver Script ---
try:
    with open(JSON_FILE_PATH, 'r') as f:
        manufacturer_models = json.load(f)
    print(f"Successfully loaded {len(manufacturer_models)} manufacturers from JSON file.")
except FileNotFoundError:
    print(f"FATAL ERROR: Could not find JSON file at '{JSON_FILE_PATH}'")
    exit()

try:
    options = Options()
    options.add_argument("start-maximized")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    service = Service(ChromeDriverManager().install())
    wd = webdriver.Chrome(service=service, options=options)
except Exception as e:
    print(f"FATAL ERROR: Could not start Selenium webdriver. Error: {e}")
    exit()

if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)

try:
    for manufacturer, models in manufacturer_models.items():
        print(f"\n--- Processing Manufacturer: {manufacturer} ---")
        manufacturer_dir = os.path.join(BASE_DIR, manufacturer)
        if not os.path.exists(manufacturer_dir):
            os.makedirs(manufacturer_dir)

        for model in models:
            sanitized_model_name = model.replace('/', '_').replace('\\', '_')
            final_image_path = os.path.join(manufacturer_dir, f"{sanitized_model_name}.jpg")

            if os.path.exists(final_image_path):
                print(f"Skipping '{model}', it already exists.")
                continue

            query = f"{manufacturer} {sanitized_model_name} car"
            print(f"Processing model: {model} (Query: '{query}')")
            
            get_and_download_first_image(wd, query, final_image_path)
            
            time.sleep(1) # Polite delay to avoid getting blocked
            
finally:
    print("\n✅ Image downloading process finished. Closing browser.")
    wd.quit()

Successfully loaded 51 manufacturers from JSON file.

--- Processing Manufacturer: Acura ---
Processing model: EL (Query: 'Acura EL car')
Cookie consent button found. Clicking it...
-> FAILED: An unexpected error occurred for query 'Acura EL car': Message: 
Stacktrace:
	GetHandleVerifier [0x0x119c333+65459]
	GetHandleVerifier [0x0x119c374+65524]
	(No symbol) [0x0xfbd973]
	(No symbol) [0x0x10076e7]
	(No symbol) [0x0x1007a8b]
	(No symbol) [0x0x104dea2]
	(No symbol) [0x0x1029e44]
	(No symbol) [0x0x104b606]
	(No symbol) [0x0x1029bf6]
	(No symbol) [0x0xffb38e]
	(No symbol) [0x0xffc274]
	GetHandleVerifier [0x0x141eda3+2697763]
	GetHandleVerifier [0x0x1419ec7+2677575]
	GetHandleVerifier [0x0x11c4194+228884]
	GetHandleVerifier [0x0x11b49f8+165496]
	GetHandleVerifier [0x0x11bb18d+192013]
	GetHandleVerifier [0x0x11a47d8+99416]
	GetHandleVerifier [0x0x11a4972+99826]
	GetHandleVerifier [0x0x118ebea+10346]
	BaseThreadInitThunk [0x0x75fc5d49+25]
	RtlInitializeExceptionChain [0x0x777cd6db+107]
	RtlGe

KeyboardInterrupt: 

In [ ]:
# Import packages

import selenium
from selenium import webdriver
import requests
import shutil
import hashlib
import os
import io
import time
from PIL import Image

# Define the path to chrome driver
DRIVER_PATH = 'C:\\Users\\Anwender\\chromedriver_win32\\chromedriver'
wd = webdriver.Chrome(executable_path = DRIVER_PATH)
# This searches images from google.com
wd.get('https://google.com')

# Create the scraper class
class GoogleScraper():
    '''Downloades images from google based on the query.
       webdriver - Selenium webdriver
       max_num_of_images - Maximum number of images that we want to download
    '''
    def __init__(self, webdriver:webdriver, max_num_of_images:int):
        self.wd = webdriver
        self.max_num_of_images = max_num_of_images

    def _scroll_to_the_end(self):
        wd.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)  

    def _build_query(self, query:str):
        return f"https://www.google.com/search?safe=off&site=&tbm=isch&source=hp&q={query}&oq={query}&gs_l=img"

    def _get_info(self, query: str):
        image_urls = set()

        wd.get(self._build_query(query))
        self._scroll_to_the_end()

        # img.Q4LuWd is the google tumbnail selector
        thumbnails = self.wd.find_elements_by_css_selector("img.Q4LuWd")

        print(f"Found {len(thumbnails)} images...")
        print(f"Getting the links...")

        for img in thumbnails[0:self.max_num_of_images]:
            # We need to click every thumbnail so we can get the full image.
            try:
                img.click()
            except Exception:
                print('ERROR: Cannot click on the image.')
                continue

            images = wd.find_elements_by_css_selector('img.n3VNCb')
            time.sleep(0.3)

            for image in images:
                if image.get_attribute('src') and 'http' in image.get_attribute('src'):
                    image_urls.add(image.get_attribute('src'))

        return image_urls

    def download_image(self, folder_path:str, url:str):
        try:
            image_content = requests.get(url).content

        except Exception as e:
            print(f"ERROR: Could not download {url} - {e}")

        try:
            image_file = io.BytesIO(image_content)
            image = Image.open(image_file).convert('RGB')
            file = os.path.join(folder_path,hashlib.sha1(image_content).hexdigest()[:10] + '.jpg')

            with open(file, 'wb') as f:
                image.save(f, "JPEG", quality=85)
            print(f"SUCCESS: saved {url} - as {file}")

        except Exception as e:
            print(f"ERROR: Could not save {url} - {e}")

    def scrape_images(self, query:str, folder_path= 'C:\\Users\\Username\\web_scrape'):
        folder = os.path.join(folder_path,'_'.join(query.lower().split(' ')))

        if not os.path.exists(folder):
            os.makedirs(folder)

        image_info = self._get_info(query)
        print(f"Downloading images...")

        for image in image_info:
            self.download_image(folder, image)

# Run scrape command
gs = GoogleScraper(wd, 100)
gs.scrape_images('fufu meal')

Downloading: Acura MDX car
[%] Downloading Images to C:\Users\Anwender\Downloads\Master\data_ui\images\Acura\Acura MDX car


[!!]Indexing page: 1

[%] Indexed 47 Images on Page 1.


[%] Downloading Image #1 from https://static1.topspeedimages.com/wordpress/wp-content/uploads/2024/09/2024-acura-integra-type-s-parked-on-a-highway.jpg
[%] File Downloaded !



[%] Done. Downloaded 1 images.
Downloading: Acura RDX car
[%] Downloading Images to C:\Users\Anwender\Downloads\Master\data_ui\images\Acura\Acura RDX car


[!!]Indexing page: 1

[%] Indexed 47 Images on Page 1.


[%] Downloading Image #1 from https://static1.topspeedimages.com/wordpress/wp-content/uploads/2024/09/2024-acura-integra-type-s-parked-on-a-highway.jpg
[%] File Downloaded !



[%] Done. Downloaded 1 images.
Downloading: Acura TLX car
[%] Downloading Images to C:\Users\Anwender\Downloads\Master\data_ui\images\Acura\Acura TLX car


[!!]Indexing page: 1

[%] Indexed 47 Images on Page 1.


[%] Downloading Image #1 from https://

FileExistsError: [WinError 183] Eine Datei kann nicht erstellt werden, wenn sie bereits vorhanden ist: 'C:/Users/Anwender/Downloads/Master/data_ui/images\\BMW\\BMW 530I car\\Image_1.jpg' -> 'C:/Users/Anwender/Downloads/Master/data_ui/images\\BMW\\530I.png'

In [11]:
df.columns

Index(['year_of_manufacture', 'manufacturer', 'model', 'mileage', 'body_type',
       ' Engine', ' Transmission', ' Drivetrain', ' Exterior Colour',
       ' Interior Colour', ' Passengers', ' Doors', ' Fuel Type', ' City',
       ' Highway', 'price'],
      dtype='object')